In [ ]:
from pathlib import Path
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.utils import load_img, img_to_array

V2_MODEL_PATH = Path("/content/mobilenetv2_crack_final.keras")

print("Model exists:", V2_MODEL_PATH.exists())
print("Model path:", V2_MODEL_PATH)

v2_model = tf.keras.models.load_model(V2_MODEL_PATH)

class_names = ["Negative", "Positive"]

print("Loaded MobileNetV2 model successfully.")
print("Class names:", class_names)

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

uploaded_image_path = Path(list(uploaded.keys())[0])

print("Uploaded image:", uploaded_image_path)

In [ ]:
def predict_uploaded_image_v2(
    img_path,
    model,
    threshold=0.5
):
    img = load_img(img_path, target_size=(224, 224))
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)

    start_time = time.perf_counter()
    prob_positive = float(model.predict(arr, verbose=0)[0][0])
    inference_time = time.perf_counter() - start_time

    if prob_positive >= threshold:
        pred_label = "Positive"
        meaning = "Có vết nứt"
        confidence = prob_positive
    else:
        pred_label = "Negative"
        meaning = "Không có vết nứt"
        confidence = 1 - prob_positive

    return {
        "image_path": str(img_path),
        "pred_label": pred_label,
        "meaning": meaning,
        "prob_positive": prob_positive,
        "confidence": confidence,
        "inference_time_seconds": inference_time,
        "threshold": threshold
    }

In [ ]:
result = predict_uploaded_image_v2(
    img_path=uploaded_image_path,
    model=v2_model,
    threshold=0.5
)

print("===== KẾT QUẢ DỰ ĐOÁN MOBILE-NET V2 =====")
print("Ảnh:", result["image_path"])
print("Nhãn dự đoán:", result["pred_label"])
print("Ý nghĩa:", result["meaning"])
print("Xác suất Positive / Có vết nứt:", f"{result['prob_positive'] * 100:.2f}%")
print("Độ tin cậy:", f"{result['confidence'] * 100:.2f}%")
print("Thời gian suy luận:", f"{result['inference_time_seconds']:.4f} giây")
print("Threshold:", result["threshold"])

img_show = load_img(uploaded_image_path)

plt.figure(figsize=(6, 6))
plt.imshow(img_show)
plt.axis("off")
plt.title(
    f"{result['meaning']}\n"
    f"Confidence: {result['confidence']:.2%} | "
    f"Positive prob: {result['prob_positive']:.2%}"
)
plt.show()